# Dự báo giá nhà TP.HCM — chạy thử end-to-endNotebook này đi qua trọn đường ống bằng **dữ liệu đã có sẵn** trong `data/processed/`.Nó không crawl lại và không huấn luyện lại toàn bộ danh mục mô hình: mục đích là đểngười đọc thấy từng bước làm gì, trên dữ liệu thật, trong vài phút.Muốn chạy lại từ đầu (kể cả bước thu thập), xem `docs/huong-dan-su-dung.md`.**Chạy trước:** `make prep` để có `data/processed/listings.parquet`.

## 1. Nạp dữ liệu đã làm sạch

In [ ]:
import sys, pathlibsys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()))import pandas as pdfrom src import configdf = pd.read_parquet(config.DATA_PROCESSED / "listings.parquet")print(f"{len(df):,} dòng, {df.shape[1]} cột")df.groupby("source").agg(    so_tin=("listing_id", "size"),    gia_trung_vi_ty=("total_price_vnd", lambda s: round(s.median() / 1e9, 2)),    dien_tich_trung_vi=("area_m2", "median"),)

## 2. Bộ chuẩn hoá giáBa dạng phải phân biệt: tổng giá, đơn giá theo m² (phải nhân diện tích), và "thoả thuận"— là **nhãn thiếu**, bị loại chứ không được điền.

In [ ]:
from src.preprocess.price import parse_pricefor text in ["5 tỷ 2", "5,2 tỷ", "5200 triệu", "52 triệu/m²", "6.79 tỷ", "thoả thuận"]:    parsed = parse_price(text, area_m2=100)    gia = f"{parsed.total_vnd/1e9:.2f} tỷ" if parsed.total_vnd else "— (nhãn thiếu)"    print(f"{text!r:16} → {gia:>16}   [{parsed.kind}]")

## 3. Trích đặc trưng định lượng từ mô tảBảy trường rút bằng regex. Chú ý luật cộng tầng của tiếng Việt: "1 trệt 2 lầu" = 3 tầng.

In [ ]:
from src.preprocess.extract import extract_alltin = df.loc[df["description"].str.len() > 400, "description"].iloc[0]print(tin[:400], "...\n")for k, v in extract_all(tin).as_dict().items():    print(f"  {k:16} {v}")

## 4. Xoá dấu vết giá khỏi văn bảnNếu để nguyên, mô hình không học định giá — nó học đọc lại con số trong mô tả.Cột `description_tokens` là bản đã lọc giá **và** đã tách từ tiếng Việt.

In [ ]:
from src.preprocess.leakage import count_money_tokensgoc = df["description"].iloc[0]sach = df["description_tokens"].iloc[0]print("GỐC :", goc[:180].replace("\n", " "), "\n")print("SẠCH:", sach[:180], "\n")print("Số dòng còn cụm tiền trong toàn bộ tập:",      int(df["description_tokens"].map(lambda t: count_money_tokens(t) > 0).sum()), "/", len(df))

## 5. Ma trận đặc trưng`ColumnTransformer` ba nhánh: số / phân loại / văn bản. Mọi bước fit nằm **trong**pipeline nên không có thống kê nào bị tính trên tập kiểm tra.

In [ ]:
from src.features.build import build_feature_frame, build_pipelineX = build_feature_frame(df)print(f"bảng đặc trưng: {X.shape[0]:,} dòng × {X.shape[1]} cột")pipeline = build_pipeline()M = pipeline.fit_transform(X.head(3000))print(f"sau ColumnTransformer: {M.shape[1]} cột")print("nhánh:", [name for name, _, _ in pipeline.transformers])

## 6. Huấn luyện nhanh một mô hìnhDùng đúng phép chia đã lưu ra đĩa để con số ở đây so được với bảng kết quả trong báo cáo.

In [ ]:
from src.evaluation.metrics import compute_metricsfrom src.evaluation.runner import build_estimatorfrom src.evaluation.splits import make_splitsfrom src.models.registry import build_registrysub = df[df["source"] == "chotot"].reset_index(drop=True)split = make_splits(sub, "e1_chotot")Xs, y = build_feature_frame(sub), sub["total_price_vnd"].to_numpy(float)spec = next(s for s in build_registry() if s.name == "Random Forest")model = build_estimator(spec).fit(Xs.iloc[split["train"]], y[split["train"]])for k, v in compute_metrics(y[split["test"]], model.predict(Xs.iloc[split["test"]])).items():    print(f"  {k:12} {v:8.3f}")

## 7. So với baseline "môi giới"Trung vị giá/m² theo (quận, phường, loại nhà) nhân diện tích. Đây mới là mốc đáng quantâm: nếu mô hình chỉ nhỉnh hơn cách tính này vài phần trăm thì giá trị nằm ở dữ liệu địabàn chứ không nằm ở mô hình.

In [ ]:
cols = ["district", "ward", "property_type", "area_m2"]baseline = build_estimator(next(s for s in build_registry() if s.needs_raw_frame))baseline.fit(sub.loc[split["train"], cols], y[split["train"]])for k, v in compute_metrics(y[split["test"]], baseline.predict(sub.loc[split["test"], cols])).items():    print(f"  {k:12} {v:8.3f}")

## 8. Đọc tiếp ở đâu| Nội dung | Đường dẫn ||---|---|| Bảng so sánh đầy đủ 12 mô hình | `reports/tables/e1-results-*.md` || Chất lượng trích xuất từ văn bản | `reports/tables/extraction-quality.md` || Dòng dữ liệu qua từng bước làm sạch | `reports/tables/data-funnel.md` || Kiến trúc mã nguồn và mô tả từng hàm | `reports/technical-report/` || Web app dự báo | `make demo` |